# Impact of real life events on YouTube gaming views

In this section, we will examine how **real-world events** related to gaming **influence** viewership trends on YouTube. It is reasonable to assume that gaming channels capitalize on popular trends and generate content around **trending topics** to maximize views. To validate this assumption, we will analyze viewership patterns for individual games over time and compare them with significant events such as **e-sports tournaments, game releases, and movie premieres**.

In [77]:
# data frames
import pandas as pd
import polars as pl

# utils
import sys
sys.path.insert(0, '../')
from src.utils import *
from pathlib import Path

# progress tracking
from tqdm import tqdm
tqdm.pandas()

# turn off warnings
import warnings
warnings.filterwarnings("ignore")

src_path = Path('../src').resolve()  
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from timeseries_utils import *

# youniverse paths
VIDEOS_PATH = "../data/youniverse/filtered/gaming_videos.tsv"
CHANNELS_PATH =  "../data/youniverse/filtered/gaming_channels.tsv"
TIMESERIES_PATH = "../data/youniverse/filtered/gaming_timeseries.tsv"

# additional paths
GAMES_PATH = "../data/games.csv"
ESPORTS_PATH = "../data/esports_tournaments.csv"
WORDS_PATH = "../data/words_alpha.txt"

# random seed
RANDOM_STATE = 1

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Time-series data

In [3]:
timeseries_df = pl.read_csv(TIMESERIES_PATH, separator='\t')
videos_df = pl.read_csv(VIDEOS_PATH, separator='\t')
print(f"We have a total of {len(videos_df)} videos")


We have a total of 13720303 videos


In [4]:
videos_sample_df = videos_df.sample(fraction=1, seed=RANDOM_STATE)
print(f'There are {len(videos_sample_df)} videos in the sample dataset.')

There are 13720303 videos in the sample dataset.


### Video games dataset

 Now, we will cross our dataset with another one, containing an almost-comprehensive list of more than **41k** unique commercial video games. It is available [here](https://www.kaggle.com/datasets/matheusfonsecachaves/popular-video-games). So as to keep this study manageable, we will only focus on the first **~1k** most popular games, since we assume they represent the vast majority of YouTube gaming videos.
 
Let's load and filter out the dataset.

In [5]:
games_df = pd.read_csv(GAMES_PATH, index_col=0).drop_duplicates("Title").reset_index(drop=True)

We'll use the `Plays` feature to estimate the popularity of the games, and we'll use a cutoff of **2k** games to keep.

In [6]:
cutoff = 2000
games_df["Plays_Numeric"] = games_df["Plays"].apply(lambda x: float(x.replace('k', '').replace('K', '')) * 1000 if "k" in x or "K" in x else float(x))
games_df = games_df.sort_values(by="Plays_Numeric", ascending=False).drop(columns=["Plays_Numeric"])
games_df = games_df.iloc[:cutoff]

Then, we'll remove game names that are too short and those that are common english names, as they are likely to be noise. We pick the list of common english names [here](https://github.com/dwyl/english-words).

In [7]:
with open(WORDS_PATH, "r") as f:
    words = {line.strip() for line in f}

games_df = games_df[(games_df["Title"].str.len() > 4) & ~(games_df["Title"].str.lower().isin(words))].reset_index(drop=True)

Since we're not interested in specific versions but rather in the game itself, we will remove all games that are a superstring of another game.

In [8]:
games_df = games_df[games_df['Title'].apply(lambda x: not any(other in x for other in games_df['Title'] if x != other))].reset_index(drop=True)

Since the titles and tags are quite noisy and may contain a lot of irrelevant information as well as different names for a given video game (e.g. *LoL* for *League of Legends*, *gta* for *Grand Theft Auto V*, etc.), we searched for a good and effective way to extract the game names from the `title` and `tags` field. 

We lowercase and remove punctuation from all video games names, titles and tags, and then we start by looking if a game name is entirely contained in the video title. If it is, we assign the video to this game. If it is not, we look if a game name is entirely contained in one of the tags. If there is only one game name, we assign the video to this game. Otherwise, if there is no game or several games in the tags, we do not assign the video. This way, we can assign a game to **~50%** of the videos.

We'll only process the previously devised sample (`pandas_sample_df`) of $5\%$ of the data.

In [9]:
# ~20s
videos_sample_df = videos_sample_df.with_columns(
    pl.col("title").map_elements(preprocess_name),
    pl.col("tags").map_elements(preprocess_name)
)
game_titles = games_df["Title"].apply(preprocess_name).tolist()

In [10]:
# ~1min40s
videos_with_game_df = videos_sample_df.with_columns(
    [
        pl.struct(["title", "tags"])
        .map_elements(
            w_pbar(
                tqdm(total=len(videos_sample_df)),
                lambda row: map_to_game(row["title"], row["tags"], game_titles),
            ),
            return_dtype=str,
        )
        .alias("video_game")
    ]
).drop_nulls(subset="video_game")

100%|██████████| 13720303/13720303 [1:10:57<00:00, 3222.77it/s] 


In [28]:
videos_with_game_df.to_pandas().to_csv("videos_with_game_df.csv", index=False)
#videos_with_game_df = pd.read_csv("processed_dataframe.csv")

In [29]:
print(f'Percentage of classified games over the sample : {len(videos_with_game_df) / len(videos_sample_df) * 100:.2f}%')

Percentage of classified games over the sample : 49.72%


Almost half of our sample was assigned a game ! We can now explore what are the top-15 most popular games in our dataset !

In [30]:
game_percentage(videos_with_game_df, channels=False).head(10)

video_game,percentage
str,f64
"""minecraft""",11.23
"""fortnite""",5.53
"""call of duty""",5.4
"""league of legends""",3.46
"""roblox""",3.02
"""grand theft auto""",3.01
"""dota 2""",2.19
"""super smash bros""",1.36
"""final fantasy""",1.18


Unsurprisingly, YouTube is dominated by *minecraft*, *fortnite*, *call of duty*, *league of legends*, ... We'll create a subset of our channels with their top video game assigned to them.

In [31]:
channels_top_game_df = (
    videos_with_game_df
    .group_by(["channel_id", "video_game"])
    .agg(pl.count().alias("play_count"))
    .sort(["channel_id", "play_count"], descending=True)
    .group_by("channel_id")
    .agg(pl.col("video_game").first().alias("top_game"))
)

Let's now explore some statistics about the channels and tehir main video game.
We will see which games are **the most frequently** used as **main game** in our sample of **channels**.

In [93]:
channels_top_game_df

channel_id,top_game
str,str
"""UCbbwieYl0WBCPsXB9uKvVUA""","""borderlands 2"""
"""UCSaUFXLe3d9WqGp7TA4FKZA""","""fortnite"""
"""UCZ3MT9dfp3_6UeXSZ2zEekA""","""grand theft auto"""
"""UCSEm-Eis0XN1PiQzy3HAetw""","""clash of clans"""
"""UCQ0KFtUTiUVKL9TDb8m65NQ""","""league of legends"""
…,…
"""UC15LrSNB79lV3QPwwvoCx0A""","""team fortress 2"""
"""UC1u38xXqNp3FLkEG7uszxGg""","""fifa 19"""
"""UCVC5-Y6ez4sk3mtbUmN-SUQ""","""fortnite"""


In [98]:
def game_percentage(df: pl.DataFrame):
    """
    Returns the percentage of videos games associated with channels or videos
    Args:
        df(pl.DataFrame): videos or channels dataframe
    Returns:
        with_game_counts(pl.DataFrame): the percentage of videos games associated with channels or videos
    """

    column_name = "video_game"
    with_game_counts = df.get_column(column_name).value_counts().sort("count", descending=True)
    with_game_counts = with_game_counts.with_columns(
        (100 * pl.col("count") / len(df)).alias("percentage")
    )

    with_game_counts = with_game_counts.with_columns(
        pl.col("percentage").round(2)
    ).drop("count").sort("percentage", descending=True)
    return with_game_counts

popular_games = game_percentage(videos_with_game_df).head(10)["video_game"].to_list()

In [101]:
channels_top_game_df

channel_id,top_game
str,str
"""UCbbwieYl0WBCPsXB9uKvVUA""","""borderlands 2"""
"""UCSaUFXLe3d9WqGp7TA4FKZA""","""fortnite"""
"""UCZ3MT9dfp3_6UeXSZ2zEekA""","""grand theft auto"""
"""UCSEm-Eis0XN1PiQzy3HAetw""","""clash of clans"""
"""UCQ0KFtUTiUVKL9TDb8m65NQ""","""league of legends"""
…,…
"""UC15LrSNB79lV3QPwwvoCx0A""","""team fortress 2"""
"""UC1u38xXqNp3FLkEG7uszxGg""","""fifa 19"""
"""UCVC5-Y6ez4sk3mtbUmN-SUQ""","""fortnite"""


In [107]:
export_views_top_games_json(
    df=videos_with_game_df,
    game_names= popular_games,
    output_file="weekly_delta_views.json")

## Impact of real world events on channels

Each channel is assigne to a video game that we will consider as its main. Our question here is: **How does a channel's subscribers count change when an event related to its main video game occurs?**

Let's choose some **mainstream games** associated to popular events to observe this behaviour. 



In [78]:
games_timeseries_df = games_timeseries(channels_top_game_df, timeseries_df)

In [90]:
plot_metric(games_timeseries_df, game_names=["fortnite","minecraft"], period="M")

An interesting game is **Mortal Kombat**: with the release of its 11th edition in 2019 the game generated great anticipation

In [79]:
mk_events =[("Mortal Kombat 11 Release", "2019-04-23")]
plot_metric(games_timeseries_df, game_names=["mortal kombat"], dates=mk_events, period="M", metric="subs")
metric_to_json(games_timeseries_df, game_names=["mortal kombat"], dates=mk_events, period="M", output_file="subs_mk.json")

In [80]:
smash_bros_events = [("Super Smash Bros Ultimate Release", "2018-12-07")]
plot_metric(games_timeseries_df, game_names =["super smash bros"], dates=smash_bros_events, period="W", window=7, metric="subs")
metric_to_json(games_timeseries_df, game_names=["super smash bros"], dates=smash_bros_events, period="W", window=7, output_file="subs_smash.json")

## Views generated by videos given a certain game

In this section, our focus will shift from channels to **individual videos**. Each video is linked to a specific video game, allowing us to analyze how **video** content related to a particular game **generates views over time**. We will also compare these trends to relevant real-world events for deeper insights. Obviously since we only take a sample of the videos dataset the cumulative number of views will be much smaller. What really interests us here is the **trends** we see in the graphs notably the **spikes** and the **troughs**.

We'll start with **League of Legends (LoL)** that has world championships every year.

In [87]:
lol_events = [
    ("World Championship 2014", "2014-09-18"),
    ("World Championship 2016", "2016-09-29"),
    ("World Championship 2017", "2018-10-02"),

]
plot_metric(videos_with_game_df, game_names=["league of legends"],dates=lol_events ,channel = False, window = 10, metric="views")
metric_to_json(df=videos_with_game_df, game_names=["league of legends"], dates=lol_events, output_file="views_lol.json", metric="views",channel_views=False, lower_cutoff="2014-07-01") 

We can clearly observe a rise in views and interest for League of Legends channels around world championship periods.

One extra example is a popular game franchise called **Assasin's creed** who has reoccuring game releases.

In [88]:
ac_events = [
    ("AC origins release", "2017-10-27"),
    ("AC Rogue release", "2014-11-11"),
    ("AC Odyssey", "2018-10-02"),
]
plot_metric(videos_with_game_df, game_names=["assassins creed"],dates=ac_events,channel = False, window = 10)
metric_to_json(df=videos_with_game_df, game_names=["assassins creed"], dates=ac_events, output_file="views_ac.json", metric="views", channel_views=False, lower_cutoff="2014-09-01")

In [89]:
battle_royale_release = ("Battle royale mode release", "2017-09-26")
fortnite_events = [battle_royale_release]
plot_metric(videos_with_game_df, game_names=["fortnite"],dates=fortnite_events,channel = False, window = 10)
metric_to_json(df=videos_with_game_df, game_names=["fortnite"], dates=fortnite_events, output_file="fortnite.json", channel_views=False)

Fortnite is one of the most successful video games of the last 10 years. It became particularly **popular** after the **release of its Battle Royale** mode, which we can clearly observe in the graph.

In [85]:
fifa_events =  [("FIFA 14 release", "2013-09-24"),("FIFA 15 Release", "2014-09-23"),("FIFA 16 Release", "2015-09-22"),("FIFA 17 Release", "2016-09-27"),("FIFA 18 Release", "2017-09-29")]
plot_metric(videos_with_game_df, game_names=["fifa 14","fifa 15","fifa 16","fifa 17","fifa 18"],channel=False ,dates=fifa_events, window = 4)
metric_to_json(df=videos_with_game_df, game_names=["fifa 14","fifa 15","fifa 16","fifa 17","fifa 18"], dates=fifa_events, output_file="fifa_14_to_18.json", channel_views=False, lower_cutoff="2013-03-01")

The fifa franchise is also a fan favorite with one iteration of the game released per year. We can clearly see that the interest in fifa 15 after its release and falls quickly when fifa 16 is out.

In [86]:
final_fantasy_events = [("Final Fantasy XV Release", "2016-11-29")]
plot_metric(videos_with_game_df, game_names=["final fantasy"],dates=final_fantasy_events, channel=False, window = 6)
metric_to_json(df=videos_with_game_df, game_names=["final fantasy"], dates=final_fantasy_events, output_file="views_ff.json", metric="views", channel_views=False, lower_cutoff="2016-02-01")